<a href="https://colab.research.google.com/github/tlgus9989-art/bigdataHackathon/blob/main/%EA%B5%90%EC%88%98%EB%8B%98_%ED%8F%89%EA%B0%80%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교수님 실행용: 최종 CNN 학습부터 평가까지

기존 Colab이나 저장 가중치에 접근하지 않습니다. 이 노트북의 새 런타임에서 **Kaggle URL 다운로드 → 중복 제거·분할 → deep_low_lr 하나 학습 → 최적 회차 복원 → 시험 평가**를 수행합니다.

Colab에서 **런타임 > 런타임 유형 변경 > T4 GPU**를 선택하고 위에서부터 실행하세요. 데이터 다운로드를 위한 인터넷 연결이 필요합니다. Google Drive 연결, Kaggle 데이터 수동 업로드, 기존 `.pt` 업로드는 필요하지 않습니다.

이전 다섯 설정 중 최소 검증 손실을 기록한 설정: 합성곱 3개, BatchNorm, ReLU, Dropout 0.3, Adam 학습률 0.0003, 배치 64, 회전·좌우반전 증강. 최대 25회, 검증 손실 4회 미개선 시 종료합니다. 이전 결과는 14회차 선택·시험 정확도 80.07%였지만 이번 실행의 선택 회차와 수치는 새 학습 결과로 결정합니다.

모델·학습·평가는 PyTorch/torchvision만 사용합니다. 추가 데이터와 사전학습 모델은 사용하지 않습니다. 재학습 결과는 데이터 버전·GPU·라이브러리 환경에 따라 달라질 수 있습니다.

### 기존 결과 재현 조건
기존 CNN 클래스와 학습 함수를 그대로 복원했습니다. 데이터 정렬·중복 제거·분할, 시드 12345, 모델 생성 전 시드 재설정, 배치 순서용 Generator, 증강 순서, 평가 호출 순서, Adam 설정과 조기 종료 조건을 유지했습니다. 다섯 모델을 다시 학습할 필요는 없습니다. 기존 코드도 각 모델 학습 직전에 시드를 재설정했기 때문입니다.

기존 실행의 정확한 PyTorch·torchvision·CUDA 버전과 GPU 정보는 제공되지 않았습니다. 동일 수치를 보장할 수 없으며, 80.07%를 결과로 강제 출력하거나 그 수치가 나올 때까지 시험셋으로 모델을 고르지 않습니다. 기존 실행과 알고리즘을 바꿀 수 있는 결정적 연산 강제 설정은 새로 추가하지 않았습니다. 아래에서 이번 실행 환경을 기록합니다.


## 1. Kaggle URL에서 다운로드 및 압축 해제

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
import hashlib
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_URL = "https://www.kaggle.com/api/v1/datasets/download/mostafaabla/garbage-classification"
ZIP_PATH = Path("/content/archive.zip")
print("Kaggle 데이터 다운로드 중...")
urlretrieve(DATA_URL, ZIP_PATH)
print("다운로드 완료")
work = Path(mkdtemp(prefix="trash_", dir="/content"))
with ZipFile(ZIP_PATH) as archive:
    archive.extractall(work)

root = next(p.parent for p in work.rglob("paper") if (p.parent / "plastic").is_dir())
source = datasets.ImageFolder(root)
assert len(source.classes) == 12, source.classes
seed = 12345
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, root, len(source), source.classes)
print("torch:", torch.__version__)

import torchvision
import platform
runtime_info = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__), "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled()
}
print("이번 실행 환경:", runtime_info)


## 2. 중복 제거 및 학습·검증·시험 분할
기존 실험과 동일하게 시드 12345로 클래스별 약 70:15:15로 분할합니다. 파일 바이트가 같은 중복만 제거합니다. 시험셋은 학습·최적 회차 선택에 사용하지 않습니다.

In [ ]:
from collections import defaultdict

by_hash = defaultdict(list)
for idx, (path, label) in enumerate(source.samples):
    digest = hashlib.sha256(Path(path).read_bytes()).digest()
    by_hash[digest].append((idx, label))

by_class = defaultdict(list)
for group in by_hash.values():
    if len({label for _, label in group}) == 1:
        idx, label = group[0]
        by_class[label].append(idx)

rng = torch.Generator().manual_seed(seed)
train_ids, val_ids, test_ids = [], [], []
print("클래스 | 학습 | 검증 | 시험")
for label, name in enumerate(source.classes):
    ids = by_class[label]
    ids = [ids[i] for i in torch.randperm(len(ids), generator=rng).tolist()]
    n_val = n_test = int(len(ids) * 0.15)
    n_train = len(ids) - n_val - n_test
    train_ids.extend(ids[:n_train])
    val_ids.extend(ids[n_train:n_train + n_val])
    test_ids.extend(ids[n_train + n_val:])
    print(name, n_train, n_val, n_test)
print("중복·라벨 충돌로 제외:", len(source) - len(train_ids) - len(val_ids) - len(test_ids))

## 3. 전처리·증강 및 결과 폴더
RGB 64×64, ToTensor. 학습에만 ±10도 회전과 좌우반전을 적용합니다. 학습 정확도 측정·검증·시험에는 증강을 적용하지 않습니다.

In [ ]:
from datetime import datetime
import json

run_dir = Path("/content") / ("trash_course_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
run_dir.mkdir()

plain_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])
aug_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
plain_data = datasets.ImageFolder(root, transform=plain_tf)
aug_data = datasets.ImageFolder(root, transform=aug_tf)

train_eval_loader = DataLoader(Subset(plain_data, train_ids), batch_size=64, shuffle=False)
val_loader = DataLoader(Subset(plain_data, val_ids), batch_size=64, shuffle=False)
test_loader = DataLoader(Subset(plain_data, test_ids), batch_size=64, shuffle=False)

splits = {"train": train_ids, "val": val_ids, "test": test_ids}
manifest = {name: [{"path": str(Path(source.samples[i][0]).relative_to(root)),
                    "label": source.samples[i][1]} for i in ids] for name, ids in splits.items()}
(run_dir / "splits.json").write_text(json.dumps(
    {"classes": source.classes, "seed": seed, "splits": manifest},
    ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 폴더:", run_dir)
(run_dir / "runtime_info.json").write_text(
    json.dumps(runtime_info, ensure_ascii=False, indent=2), encoding="utf-8")


## 4. 기존 실행과 동일한 CNN 정의
기존 학습 노트북의 클래스 정의를 그대로 사용합니다. 여러 구조를 정의할 수 있지만 실제로 학습하는 것은 deeper=True인 deep_low_lr 하나뿐입니다.

In [ ]:
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, n_classes, dropout=0.0, batchnorm=False, deeper=False):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.batchnorm = batchnorm
        if batchnorm:
            self.bn1 = nn.BatchNorm2d(32)
            self.bn2 = nn.BatchNorm2d(64)
        self.deeper = deeper
        if deeper:
            self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
            if batchnorm:
                self.bn3 = nn.BatchNorm2d(128)
            self.fc1 = nn.Linear(128 * 8 * 8, 256)
        else:
            self.fc1 = nn.Linear(64 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.conv1(x)
        if self.batchnorm:
            x = self.bn1(x)
        x = F.max_pool2d(F.relu(x), 2)     # (32, 32, 32)

        x = self.conv2(x)
        if self.batchnorm:
            x = self.bn2(x)
        x = F.max_pool2d(F.relu(x), 2)     # (64, 16, 16)

        if self.deeper:
            x = self.conv3(x)
            if self.batchnorm:
                x = self.bn3(x)
            x = F.max_pool2d(F.relu(x), 2)  # (128, 8, 8)

        x = x.flatten(1)                   # 배치 차원을 유지하며 펼치기
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(self.dropout(x))   # 클래스별 점수

## 5. 손실 및 정확도 측정

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def measure(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total

## 6. 선택 설정 하나만 학습
매 회차 검증 손실이 낮아질 때 가중치를 저장합니다. 새로 학습한 결과 중 최적 회차를 선택하며 14회차로 고정하지 않습니다. 아래 if는 오류 대비 코드가 아니라 최적 모델 저장·조기 종료에 필요한 학습 제어입니다.

In [ ]:
def train_model(name, training_data, dropout, batchnorm, lr, deeper=False):
    torch.manual_seed(seed)
    model = CNN(len(source.classes), dropout, batchnorm, deeper).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(
        Subset(training_data, train_ids), batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )
    best_loss = float("inf")
    stale = 0
    history = []
    checkpoint = run_dir / (name + ".pt")
    print("\n실험:", name)

    for epoch in range(1, 26):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_loss, train_acc = measure(model, train_eval_loader)
        val_loss, val_acc = measure(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": val_acc})
        print(f"{epoch:02d}회 | 학습 손실 {train_loss:.4f} 정확도 {train_acc:.2%} | "
              f"검증 손실 {val_loss:.4f} 정확도 {val_acc:.2%} | "
              f"정확도 차이 {(train_acc-val_acc)*100:.2f}%p")

        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            best_acc = val_acc
            stale = 0
            torch.save({"state": model.state_dict(), "config": {"name": name, "dropout": dropout, "batchnorm": batchnorm, "lr": lr, "augment": training_data is aug_data, "deeper": deeper},
                        "classes": source.classes, "epoch": epoch,
                        "val_loss": val_loss, "seed": seed}, checkpoint)
        else:
            stale += 1
        (run_dir / (name + "_history.json")).write_text(
            json.dumps(history, indent=2), encoding="utf-8")
        if stale >= 4:
            print("검증 손실이 4회 연속 개선되지 않아 학습 종료")
            break

    return {"name": name, "checkpoint": str(checkpoint),
                    "val_loss": best_loss, "val_acc": best_acc, "epoch": best_epoch}

In [ ]:
chosen = train_model("deep_low_lr", aug_data, dropout=0.3,
                     batchnorm=True, lr=0.0003, deeper=True)
print("최적 회차:", chosen["epoch"], "| 검증 손실:", chosen["val_loss"])


## 7. 학습·검증 곡선
학습 손실만 낮아지고 검증 손실이 높아지는지 확인합니다. Dropout·증강·조기 종료는 과적합을 줄이기 위한 방법이며 과적합이 없음을 보장하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt

history = json.loads((run_dir / (chosen["name"] + "_history.json")).read_text())
epochs = [row["epoch"] for row in history]
plt.plot(epochs, [row["train_loss"] for row in history], label="Train")
plt.plot(epochs, [row["val_loss"] for row in history], label="Validation")
plt.axvline(chosen["epoch"], linestyle="--", color="gray", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.show()
plt.plot(epochs, [r["train_acc"] * 100 for r in history], label="Train")
plt.plot(epochs, [r["val_acc"] * 100 for r in history], label="Validation")
plt.axvline(chosen["epoch"], linestyle="--", color="gray", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()


## 8. 이번 실행에서 저장한 최적 모델 복원 및 시험 평가
아래 torch.load는 외부 가중치가 아니라 **바로 위 학습 셀이 이 런타임에 저장한 파일**을 읽습니다. 시험 결과를 보고 추가 튜닝하지 않습니다.

In [ ]:
saved = torch.load(chosen["checkpoint"], map_location=device, weights_only=True)
config = saved["config"]
model = CNN(len(saved["classes"]), config["dropout"], config["batchnorm"], config.get("deeper", False)).to(device)
model.load_state_dict(saved["state"])
test_loss, test_acc = measure(model, test_loader)
train_loss, train_acc = measure(model, train_eval_loader)
val_loss, val_acc = measure(model, val_loader)

print("선택 모델:", config["name"], "| 선택 회차:", saved["epoch"])
print(f"학습 정확도 {train_acc:.2%} | 검증 정확도 {val_acc:.2%} | 시험 정확도 {test_acc:.2%}")
print(f"시험 손실 {test_loss:.4f}")

correct = torch.zeros(len(source.classes))
total = torch.zeros(len(source.classes))
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        for c in range(len(source.classes)):
            mask = y == c
            total[c] += mask.sum()
            correct[c] += (pred[mask] == c).sum()

recall = correct / total.clamp_min(1)
for name, hit, count, rate in zip(source.classes, correct, total, recall):
    print(f"{name:12s} {int(hit):4d}/{int(count):4d} | 재현율 {rate:.2%}")
print(f"클래스 평균 재현율 {recall[total > 0].mean():.2%}")
(run_dir / "test_result.json").write_text(json.dumps(
    {"model": config["name"], "train_acc": train_acc, "val_acc": val_acc,
     "test_acc": test_acc, "test_loss": test_loss,
     "classes": source.classes, "recall": recall.tolist()}, indent=2), encoding="utf-8")
print(f"학습-검증 정확도 차이: {(train_acc-val_acc)*100:.2f}%p")
print(f"기존 시험 정확도 80.07% | 이번 시험 정확도 {test_acc:.2%}")
print(f"기존 표시값과 차이: {test_acc*100-80.07:+.2f}%p")


## 9. 새 이미지 업로드 후 분류하기

1. 먼저 **1~8단계 학습·시험 평가가 끝날 때까지 실행**합니다.
2. 아래 함수 정의 셀을 한 번 실행합니다.
3. 그다음 `predict_uploaded_images()` 셀을 실행하면 **파일 선택** 버튼이 나타납니다.
4. 컴퓨터에서 JPG/PNG 등 사진을 선택합니다. 여러 장은 Ctrl 키를 누른 채 선택할 수 있습니다.
5. 사진별 예측 클래스와 상위 3개 Softmax 점수가 표시되고 결과 CSV가 저장됩니다.
6. 다른 사진을 보려면 **업로드 호출 셀만 다시 실행**하세요. 재학습할 필요가 없습니다. 단, 런타임이 종료되면 모델이 사라집니다.

사진 한 장에는 평가하려는 쓰레기 한 종류가 잘 보이도록 준비하세요. 여러 종류가 함께 있거나 12종에 없는 물체도 모델은 하나의 범주로 분류합니다. Softmax 점수는 정답일 확률을 보장하지 않습니다.

### 정답을 입력해 새 이미지 정확도 계산하기
분류 결과 확인 후 아래 `true_labels`에 **파일명: 실제 클래스**를 입력하세요. 예: `{"bottle.jpg": "plastic", "box.png": "cardboard"}`. 정확도는 정답을 입력한 사진에 대해서만 계산하며 정답은 모델 예측이 아니라 실제 재질·종류로 정해야 합니다.

사용 가능한 영문 클래스: battery, biological, brown-glass, cardboard, clothes, green-glass, metal, paper, plastic, shoes, trash, white-glass.


In [ ]:
def predict_uploaded_images():
    import io, csv
    from PIL import Image
    from google.colab import files
    from IPython.display import display
    photos = files.upload()
    results = []
    model.eval()
    with torch.no_grad():
        for name, data in photos.items():
            picture = Image.open(io.BytesIO(data)).convert("RGB")
            x = plain_tf(picture).unsqueeze(0).to(device)
            scores, indices = model(x).softmax(dim=1)[0].cpu().topk(3)
            label = saved["classes"][indices[0].item()]
            print("\n사진:", name, "| 예측:", label)
            for score, index in zip(scores, indices):
                print(f"{saved['classes'][index.item()]}: {score.item():.2%}")
            display(picture.copy().resize((256, 256)))
            results.append({"filename": name, "prediction": label,
                            "softmax_score": scores[0].item()})
    with open(run_dir / "new_image_predictions.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "prediction", "softmax_score"])
        writer.writeheader()
        writer.writerows(results)
    return results


In [ ]:
new_results = predict_uploaded_images()


### 선택: 업로드한 사진의 실제 정답 입력
아래 딕셔너리를 수정하고 실행하세요. 정확도 계산에 포함된 사진 수와 결과를 출력합니다. 정답을 모르는 사진은 제외하세요.

In [ ]:
# 업로드한 실제 파일명과 정답으로 채우세요.
true_labels = {
    # "bottle.jpg": "plastic",
    # "box.png": "cardboard",
}

for label in true_labels.values():
    assert label in saved["classes"], f"학습 클래스에 없는 정답: {label}"
evaluated = [r for r in new_results if r["filename"] in true_labels]
correct_count = sum(r["prediction"] == true_labels[r["filename"]] for r in evaluated)
for row in evaluated:
    truth = true_labels[row["filename"]]
    print(row["filename"], "| 정답:", truth, "| 예측:", row["prediction"],
          "| 일치:", row["prediction"] == truth)
if evaluated:
    print(f"새 이미지 정확도: {correct_count}/{len(evaluated)} = {correct_count/len(evaluated):.2%}")
else:
    print("정답을 입력한 사진이 없습니다. true_labels에 파일명과 정답을 입력하세요.")


## 10. 선택: 이번 학습 결과 내려받기
이번 실행의 가중치·분할 목록·학습 기록·시험 결과를 보관하려면 마지막 줄의 주석을 해제하세요.

In [ ]:
def download_results():
    import shutil
    from google.colab import files
    result_zip = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
    files.download(result_zip)

# download_results()


## 참고 및 실행 확인
- 데이터: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
- 수업 실습: https://github.com/zoomer1975-boop/bigdata (week08 CNN; week09 Dropout·BatchNorm·증강)
- 이 파일은 기존 실험에서 선택한 설정만 재학습하도록 구성했습니다. 하이퍼파라미터 비교 기록은 기존 학습 노트북과 보고서에 있습니다.
- 기존 보고서의 수치는 이전 실행 결과입니다. 이 노트북의 새 수치가 다르면 이전 결과를 덮어 쓴 것처럼 설명하지 말고 별도 재현 실행 결과로 구분하세요.
- 코드 구문 및 실행 순서를 점검했으며, 작성 환경에서 전체 데이터 학습을 실행하지는 않았습니다. 제출 전 Colab에서 전체 실행하고 출력을 저장하세요.
